In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression # Importing the model used for predictions.
from sklearn.model_selection import cross_val_score # Cross Validation
from sklearn.preprocessing import StandardScaler # Data scaler
from sklearn.model_selection import GridSearchCV # Model Hyperparameter tuner

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


**# ---- Loading Data and features engineering**

In [6]:
X_train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
y_train = X_train.pop("Survived")
X_test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

# Title from Name highly predictive
X_train['Title'] = X_train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
X_train['Title'] = X_train['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
X_train['Title'] = X_train['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
X_test['Title'] = X_test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
X_test['Title'] = X_test['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
X_test['Title'] = X_test['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Simplify rare titles
for df in [X_train, X_test]:
    df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Family size — SibSp and Parch combined is more useful than separate
X_train['FamilySize'] = X_train['SibSp'] + X_train['Parch'] + 1
X_train['IsAlone'] = (X_train['FamilySize'] == 1).astype(int)
X_test['FamilySize'] = X_test['SibSp'] + X_test['Parch'] + 1 
X_test['IsAlone'] = (X_test['FamilySize'] == 1).astype(int)   


# Drop irrelevant collumns
X_train = X_train.drop(['Name', 'Ticket', 'Cabin'], axis=1)
X_test = X_test.drop(['Name', 'Ticket', 'Cabin'], axis=1) 


# One Hot Encoding for Nominal Categorical Data
X_train = pd.get_dummies(X_train, columns=['Sex', 'Embarked', 'Title'], drop_first=True, dtype=float)
X_test = pd.get_dummies(X_test, columns=['Sex', 'Embarked', 'Title'], drop_first=True, dtype=float)


# Fill missing age cells with median age
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_test['Age'] = X_test['Age'].fillna(X_train['Age'].median())
X_test['Fare'] = X_test['Fare'].fillna(X_test['Fare'].median())

# Rescaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.head()


,PassengerId,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,1,3,22.0,1,0,7.2500,2,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,2,1,38.0,1,0,71.2833,2,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3,3,26.0,0,0,7.9250,1,1,0.0,0.0,1.0,1.0,0.0,0.0,0.0
3,4,1,35.0,1,0,53.1000,2,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,5,3,35.0,0,0,8.0500,1,1,1.0,0.0,1.0,0.0,1.0,0.0,0.0


**#-----Training the model**

In [7]:
# Tune Regularization Strength
params = {'C': [10, 1, 2, 5, 0.5]}
grid = GridSearchCV(LogisticRegression(max_iter=1000), params, cv=10, scoring='accuracy')
grid.fit(X_train_scaled, y_train)
model = LogisticRegression(C=grid.best_params_['C'], max_iter=1000)

print(grid.best_params_, grid.best_score_)
model.fit(X_train_scaled, y_train)


{'C': 0.5} 0.8305368289637952


LogisticRegression(C=0.5, max_iter=1000)

**#-------Predictions**

In [8]:
predictions = model.predict(X_test_scaled)
submission = pd.DataFrame({'PassengerId': X_test['PassengerId'], 'Survived': predictions})
submission.to_csv('submission.csv', index=False)